# 03. Model Storage & Staging-Capacity Gate Downloader

Evaluates Google Drive quota, Colab local NVMe disk capacity, and staging strategy before initiating the 142-shard model transfer into `/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model`.

### Step 1: Pre-Download Staging-Capacity Gate

In [ ]:
import os
import sys
import shutil

# Model Specifications
MODEL_REPO = 'mastouri/GLM-5.2-colibri-int4-g64-with-int8-mtp'
MODEL_SIZE_GIB = 399.79
MODEL_SIZE_GB = 429.28
MIN_RECOMMENDED_DRIVE_GIB = 450.0
TEMP_CHUNK_BUFFER_GIB = 3.0

# Paths
DRIVE_MODEL_DIR = '/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model'
LOCAL_DIR = '/content'

# 1. Measure Google Drive Storage
drive_target = DRIVE_MODEL_DIR if os.path.exists(DRIVE_MODEL_DIR) else '/content/drive'
try:
    d_total, d_used, d_free = shutil.disk_usage(drive_target)
    drive_free_gib = round(d_free / (1024 ** 3), 2)
    drive_total_gib = round(d_total / (1024 ** 3), 2)
except Exception as e:
    drive_free_gib = 0.0
    drive_total_gib = 0.0

# 2. Measure Colab Local Storage
try:
    l_total, l_used, l_free = shutil.disk_usage(LOCAL_DIR)
    local_free_gib = round(l_free / (1024 ** 3), 2)
    local_total_gib = round(l_total / (1024 ** 3), 2)
except Exception as e:
    local_free_gib = 0.0
    local_total_gib = 0.0

# 3. Decision Logic & Architecture Routing
drive_pass = drive_free_gib >= MIN_RECOMMENDED_DRIVE_GIB
can_full_stage_locally = local_free_gib >= (MODEL_SIZE_GIB + 10.0)

if can_full_stage_locally:
    selected_architecture = "Option A: Full Local NVMe Staging (100% weights on fast disk)"
    local_required_gib = MODEL_SIZE_GIB
else:
    selected_architecture = "Option B: Hybrid Staging & Dual-Drive Mirroring (COLI_MODEL_MIRROR)"
    local_required_gib = 15.0  # MTP head (9.3 GB) + hot cache buffer

go_decision = drive_pass and (local_free_gib >= local_required_gib)

# 4. Render Preflight Storage Gate Report
print("=" * 75)
print("         GLM-5.2 COLIBRI STAGING-CAPACITY GATE PREFLIGHT AUDIT")
print("=" * 75)
print(f"Model Repository:               {MODEL_REPO}")
print(f"Verified Model Size:            {MODEL_SIZE_GIB:.2f} GiB ({MODEL_SIZE_GB:.2f} GB decimal)")
print(f"Temporary Buffer Requirement:   {TEMP_CHUNK_BUFFER_GIB:.2f} GiB")
print("-" * 75)
print(f"Google Drive Available Space:   {drive_free_gib:.2f} GiB (Total: {drive_total_gib:.2f} GiB)")
print(f"Google Drive Required Space:    >= {MIN_RECOMMENDED_DRIVE_GIB:.2f} GiB")
print(f"Google Drive Quota Status:      {'PASS' if drive_pass else 'FAIL - Insufficient Drive Free Space'}")
print("-" * 75)
print(f"Colab Local Available Space:    {local_free_gib:.2f} GiB (Total: {local_total_gib:.2f} GiB)")
print(f"Colab Local Required Space:     {local_required_gib:.2f} GiB (for {selected_architecture.split(':')[0]})")
print(f"Selected Runtime Architecture:  {selected_architecture}")
print("=" * 75)
print(f"PREFLIGHT DECISION:             {'GO - READY FOR DOWNLOAD' if go_decision else 'NO-GO - BLOCKING CAPACITY ISSUE'}")
print("=" * 75)

if not go_decision:
    raise SystemExit("Stopping: Storage gate criteria not satisfied. Resolve Drive or Local storage before downloading.")
else:
    print("\n✓ Capacity gate passed. Ready to proceed to Step 2.")

### Step 2: Resumable Model Shard Download

In [ ]:
# Optional: Set your Hugging Face token here if using a gated repository
os.environ['HF_TOKEN'] = 'hf_your_token_here'

# Execute atomic chunk-level downloader
!python scripts/download_model.py \
  --repo "{MODEL_REPO}" \
  --target-dir "{DRIVE_MODEL_DIR}"